# Building Damage Classification using ResNet50 on xView2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install shapely opencv-python -q
!tar -xf "/content/drive/MyDrive/xview2/train_images_labels_targets.tar" -C /content/

In [ ]:
import os
import json
import cv2
from shapely import wkt
from tqdm import tqdm

image_dir = '/content/train/images'
label_dir = '/content/train/labels'
output_dir = '/content/dataset'

classes = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
for c in classes:
    os.makedirs(os.path.join(output_dir, c), exist_ok=True)

def get_bbox(polygon):
    x, y = polygon.exterior.coords.xy
    return int(min(x)), int(min(y)), int(max(x)), int(max(y))

files = [f for f in os.listdir(label_dir) if 'post_disaster' in f][:200]

for file in tqdm(files):
    with open(os.path.join(label_dir, file)) as f:
        data = json.load(f)

    img_name = file.replace('.json', '.png')
    post_img = cv2.imread(os.path.join(image_dir, img_name))
    if post_img is None:
        continue

    for obj in data['features']['xy']:
        if 'properties' not in obj or 'subtype' not in obj['properties']:
            continue

        damage = obj['properties']['subtype']
        if damage not in classes:
            continue

        poly = wkt.loads(obj['wkt'])
        x1, y1, x2, y2 = get_bbox(poly)
        crop = post_img[y1:y2, x1:x2]

        if crop.shape[0] < 20 or crop.shape[1] < 20:
            continue

        crop = cv2.resize(crop, (128, 128))
        save_path = os.path.join(output_dir, damage, f"{file.replace('.json','')}_{x1}_{y1}.png")
        cv2.imwrite(save_path, crop)

for c in classes:
    print(c, len(os.listdir(os.path.join(output_dir, c))))

In [ ]:
import shutil
import random

base_dir = '/content/dataset'
split_dir = '/content/split_dataset'
classes = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']

for split in ['train', 'val']:
    for c in classes:
        os.makedirs(os.path.join(split_dir, split, c), exist_ok=True)

for c in classes:
    files = os.listdir(os.path.join(base_dir, c))
    random.shuffle(files)
    split_idx = int(0.8 * len(files))
    train_files = files[:split_idx]
    val_files = files[split_idx:]

    for f in train_files:
        shutil.copy(os.path.join(base_dir, c, f), os.path.join(split_dir, 'train', c, f))
    for f in val_files:
        shutil.copy(os.path.join(base_dir, c, f), os.path.join(split_dir, 'val', c, f))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
train_dir = '/content/split_dataset/train'
val_dir = '/content/split_dataset/val'

img_size = 224
batch_size = 32

train_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(val_dir, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)

class_names = train_ds.classes
print(class_names)

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 4)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1

def validate_one_epoch(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1, all_labels, all_preds

In [ ]:
num_epochs = 5
best_f1 = 0.0
save_path = '/content/best_resnet50_baseline_clean.pth'

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), save_path)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f} | Val   F1: {val_f1:.4f}")
    print('-' * 60)

In [ ]:
model.load_state_dict(torch.load(save_path))
_, val_acc, val_f1, y_true, y_pred = validate_one_epoch(model, val_loader, criterion)

print("Best Validation Accuracy:", val_acc)
print("Best Validation Macro F1:", val_f1)
print(classification_report(y_true, y_pred, target_names=class_names))
print(confusion_matrix(y_true, y_pred))